In [1]:
import os
import cv2
import torch
import numpy as np
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

mtcnn = MTCNN(image_size=160, margin=0, keep_all=True, device=device)
facenet_model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

dataset_path = "dataset"

embeddings = []
labels = []

for person_name in os.listdir(dataset_path):
    person_folder = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_folder):
        continue

    print(f"Processing: {person_name}")

    for img_name in os.listdir(person_folder):
        img_path = os.path.join(person_folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        face = mtcnn(rgb)

        if face is None:
            continue

        face = face.to(device)

        with torch.no_grad():
            embedding = facenet_model(face)
            embedding = F.normalize(embedding, p=2, dim=1)

        embeddings.append(embedding.cpu().numpy()[0])
        labels.append(person_name)

known_embeddings = np.array(embeddings)
known_names = np.array(labels)

print(f"Loaded embeddings for {len(set(known_names))} persons")

C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Processing: Diaa
Processing: Osama
Processing: Tadrous
Loaded embeddings for 3 persons


In [ ]:
def recognize_face(embedding, known_embeddings, known_names, threshold=0.45):
    sims = cosine_similarity(embedding, known_embeddings)[0]

    best_idx = np.argmax(sims)
    best_score = sims[best_idx]

    if best_score > threshold:
        return known_names[best_idx], best_score
    else:
        return "Unknown", best_score


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    boxes, _ = mtcnn.detect(rgb)
    faces = mtcnn(rgb)

    if boxes is not None and faces is not None:
        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = map(int, box)

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(frame.shape[1], x2)
            y2 = min(frame.shape[0], y2)

            face_tensor = faces[i].unsqueeze(0).to(device)

            with torch.no_grad():
                embedding = facenet_model(face_tensor)
                embedding = F.normalize(embedding, p=2, dim=1)
                embedding = embedding.cpu().numpy()

            name, score = recognize_face(
                embedding, known_embeddings, known_names, threshold=0.65
            )

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            cv2.putText(
                frame,
                f"{name} ({score:.2f})",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 255, 0),
                2
            )

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
test_img_path = "test.jpeg"
img = cv2.imread(test_img_path)

if img is None:
    print("Test image not found!")
    exit()

rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

boxes, _ = mtcnn.detect(rgb)
faces = mtcnn(rgb)  

if boxes is None or faces is None:
    print("No faces detected")
    exit()

for i, box in enumerate(boxes):
    x1, y1, x2, y2 = map(int, box)

    face_tensor = faces[i].unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = facenet_model(face_tensor)
        embedding = F.normalize(embedding, p=2, dim=1)
        embedding = embedding.cpu().numpy()

    sims = cosine_similarity(embedding, known_embeddings)[0]
    best_idx = np.argmax(sims)
    best_score = sims[best_idx]

    name = "Unknown"
  
    if best_score > 0.65 :
        name = known_names[best_idx]
  

    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(
        img,
        f"{name} ({best_score:.2f})",
        (x1, y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

cv2.imshow("Face Recognition Test", img)
cv2.waitKey(0)
cv2.destroyAllWindows()